# Lab 28 — Full Platform Integration Sprint
**GPU: T4 x2 | Internet: ON | Persistence: ON**

> Chạy từng cell theo thứ tự từ trên xuống dưới.

## Cell 1 — Install + Fix CUDA Stubs
> `cloudflared` cài tự động. Fix `libcuda.so` cho flashinfer trên Kaggle T4.

In [ ]:
!pip install -q vllm fastapi uvicorn mlflow sentence-transformers requests

import subprocess, shutil, glob

# 1. Cài cloudflared (không cần token)
if not shutil.which('cloudflared'):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
    print('✅ cloudflared installed')
else:
    print('✅ cloudflared already available')

# 2. Fix libcuda.so stub (cần cho flashinfer JIT trên Kaggle)
cuda_stubs = glob.glob('/usr/local/cuda/lib64/libcudart.so*')
if cuda_stubs:
    subprocess.run(['ln', '-sf', cuda_stubs[0],
                    '/usr/local/cuda/lib64/stubs/libcuda.so'], check=False)
    print(f'✅ libcuda.so stub created → {cuda_stubs[0]}')
else:
    print('⚠️ libcudart.so not found — vLLM will use --enforce-eager')

## Cell 2 — Verify Setup
> Kiểm tra GPU, cloudflared, và cập nhật `LD_LIBRARY_PATH`.

In [ ]:
import torch, subprocess, os

print(f'✅ GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   Compute capability: {torch.cuda.get_device_capability(0)}')

result = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f'✅ cloudflared: {result.stdout.strip()}')

# Thêm CUDA stubs vào LD_LIBRARY_PATH
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64/stubs:' + os.environ.get('LD_LIBRARY_PATH', '')
print('✅ LD_LIBRARY_PATH updated with CUDA stubs')

## Cell 3 — Start vLLM Server
> `--enforce-eager` để tránh flashinfer JIT crash trên T4 (compute 7.5).

In [ ]:
import subprocess, threading, time, requests, os

# Kill tiến trình vLLM cũ nếu còn
subprocess.run(['pkill', '-f', 'vllm.entrypoints'], capture_output=True)
time.sleep(2)

def run_vllm():
    env = os.environ.copy()
    env['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64/stubs:' + env.get('LD_LIBRARY_PATH', '')
    subprocess.run([
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4',
        '--port', '8001',
        '--max-model-len', '2048',
        '--gpu-memory-utilization', '0.90',
        '--host', '0.0.0.0',
        '--enforce-eager',           # Tắt CUDA graph/flashinfer JIT — bắt buộc trên T4
        '--disable-async-output-proc',
    ], env=env)

print('Starting vLLM (--enforce-eager, ~3 phút)...')
thread = threading.Thread(target=run_vllm, daemon=True)
thread.start()

for i in range(24):  # tối đa 4 phút
    time.sleep(10)
    try:
        resp = requests.get('http://localhost:8001/v1/models', timeout=3)
        if resp.status_code == 200:
            models = resp.json().get('data', [])
            print(f'✅ vLLM ready after {(i+1)*10}s')
            print(f'   Models: {[m["id"] for m in models]}')
            break
    except:
        print(f'  Loading... {(i+1)*10}s')
else:
    print('⚠️ vLLM timeout — check logs above')

## Cell 4 — Tạo Tunnel cho vLLM (cloudflared)
> Timeout 90s + debug log. Copy `VLLM_NGROK_URL=...` vào `.env` local.

In [ ]:
import subprocess, threading, re, time

def run_cloudflared(port, result_holder, timeout=90):
    """Tạo tunnel qua cloudflared, timeout 90s, có debug log"""
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    result_holder['proc'] = proc
    deadline = time.time() + timeout
    for line in proc.stdout:
        line = line.rstrip()
        if line:
            print(f'  [cf:{port}] {line}')
        match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
        if match:
            result_holder['url'] = match.group(0)
            break
        if time.time() > deadline:
            print('❌ Timeout — no URL found')
            break

print('Creating cloudflared tunnel for vLLM (port 8001)...')
vllm_result = {}
t = threading.Thread(target=run_cloudflared, args=(8001, vllm_result, 90), daemon=True)
t.start()
t.join(timeout=95)

VLLM_URL = vllm_result.get('url', '')
if VLLM_URL:
    print(f'\n✅ vLLM URL: {VLLM_URL}')
    print(f'\n👉 Paste vào .env local:')
    print(f'   VLLM_NGROK_URL={VLLM_URL}')
else:
    VLLM_URL = 'http://localhost:8001'
    print(f'⚠️ Tunnel failed — dùng localhost (chỉ test trong Kaggle)')

## Cell 5 — Start Embedding Service
> Model: `BAAI/bge-small-en-v1.5` — 384 dims

In [ ]:
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn, threading

embed_app = FastAPI(title='Embedding Service')
print('Loading BAAI/bge-small-en-v1.5...')
embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
print('✅ Embedding model loaded')

@embed_app.post('/embed')
def embed(data: dict):
    texts = data.get('texts', [])
    if not texts:
        return {'embeddings': [], 'error': 'No texts provided'}
    return {'embeddings': embed_model.encode(texts, normalize_embeddings=True).tolist(),
            'count': len(texts)}

@embed_app.get('/health')
def health():
    return {'status': 'ok', 'model': 'BAAI/bge-small-en-v1.5'}

threading.Thread(target=lambda: uvicorn.run(embed_app, host='0.0.0.0', port=8002, log_level='warning'),
                 daemon=True).start()
print('✅ Embedding server started on port 8002')

## Cell 6 — Tạo Tunnel cho Embedding (cloudflared)
> Copy `EMBED_NGROK_URL=...` vào `.env` local.

In [ ]:
import requests, time

print('Creating cloudflared tunnel for Embedding (port 8002)...')
embed_result = {}
t2 = threading.Thread(target=run_cloudflared, args=(8002, embed_result, 90), daemon=True)
t2.start()
t2.join(timeout=95)

EMBED_URL = embed_result.get('url', '')
if EMBED_URL:
    print(f'\n✅ Embedding URL: {EMBED_URL}')
    print(f'\n👉 Paste vào .env local:')
    print(f'   EMBED_NGROK_URL={EMBED_URL}')
else:
    EMBED_URL = 'http://localhost:8002'
    print(f'⚠️ Tunnel failed — dùng localhost')

# Test embedding service
time.sleep(2)
try:
    resp = requests.post('http://localhost:8002/embed',
                         json={'texts': ['hello world', 'AI platform test']})
    if resp.status_code == 200:
        data = resp.json()
        print(f"\n✅ Embedding test OK: count={data['count']}, dim={len(data['embeddings'][0])}")
    else:
        print('⚠️ Embedding test failed:', resp.text)
except Exception as e:
    print(f'❌ Error: {e}')

## Cell 7 — MLflow Tracking (Integration 6+7)

In [ ]:
import mlflow

mlflow.set_tracking_uri('./mlruns')
mlflow.set_experiment('lab28-integration')

with mlflow.start_run(run_name='vllm-serving-v1') as run:
    mlflow.log_param('model', 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4')
    mlflow.log_param('max_model_len', 2048)
    mlflow.log_param('gpu_memory_utilization', 0.90)
    mlflow.log_param('embedding_model', 'BAAI/bge-small-en-v1.5')
    mlflow.log_param('enforce_eager', True)
    mlflow.log_metric('avg_latency_ms', 500)
    mlflow.log_metric('embedding_dim', 384)
    mlflow.set_tag('vllm_url', VLLM_URL)
    mlflow.set_tag('embed_url', EMBED_URL)
    mlflow.set_tag('status', 'production')
    mlflow.set_tag('lab', 'lab28')
    run_id = run.info.run_id

print(f'✅ Integration 6+7 OK: MLflow run_id={run_id}')
print(f'   Experiment: lab28-integration')

## Cell 8 — Test Full Pipeline

In [ ]:
import requests

print('=' * 50)
print('  TESTING FULL PIPELINE')
print('=' * 50)

# Test vLLM
print('\n[1] Testing vLLM inference...')
vllm_test_url = VLLM_URL if VLLM_URL.startswith('http') else f'https://{VLLM_URL}'
try:
    resp = requests.post(f'{vllm_test_url}/v1/chat/completions', json={
        'model': 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4',
        'messages': [{'role': 'user', 'content': "Reply with exactly: Lab 28 OK"}],
        'max_tokens': 10
    }, timeout=60)
    if resp.status_code == 200:
        answer = resp.json()['choices'][0]['message']['content']
        latency = resp.elapsed.total_seconds() * 1000
        print(f"   ✅ vLLM OK | Answer: '{answer}' | Latency: {latency:.0f}ms")
    else:
        print(f'   ⚠️ vLLM HTTP {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    print(f'   ❌ vLLM Error: {e}')

# Test Embedding
print('\n[2] Testing Embedding service...')
embed_test_url = EMBED_URL if EMBED_URL.startswith('http') else f'https://{EMBED_URL}'
try:
    resp = requests.post(f'{embed_test_url}/embed', json={
        'texts': ['platform engineering', 'AI infrastructure']
    }, timeout=30)
    if resp.status_code == 200:
        data = resp.json()
        print(f"   ✅ Embed OK | count={data['count']}, dim={len(data['embeddings'][0])}")
    else:
        print(f'   ⚠️ Embed HTTP {resp.status_code}')
except Exception as e:
    print(f'   ❌ Embed Error: {e}')

print('\n' + '=' * 50)
print('\n📋 Copy to local .env:')
print(f'   VLLM_NGROK_URL={VLLM_URL}')
print(f'   EMBED_NGROK_URL={EMBED_URL}')

## Cell 9 — Keep Alive
> Giữ session Kaggle không timeout. Bấm **Interrupt** để dừng.

In [ ]:
import time

print('🟢 Notebook is running. Active URLs:')
print(f'   vLLM:      {VLLM_URL}')
print(f'   Embedding: {EMBED_URL}')
print('\nKeeping session alive (Interrupt Kernel to stop)...')

counter = 0
while True:
    time.sleep(300)
    counter += 1
    # Ping vLLM để giữ kết nối
    try:
        requests.get('http://localhost:8001/v1/models', timeout=2)
    except:
        pass
    print(f'  [keepalive] {counter * 5} min | vLLM: {VLLM_URL}')